In [38]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

In [39]:
train_data = pd.read_csv("Kaggle_Data/train.csv")
train_data.head()


,path,participant_id,sequence_id,sign
0,train_landmark_files/26734/1000035562.parquet,26734,1000035562,blow
1,train_landmark_files/28656/1000106739.parquet,28656,1000106739,wait
2,train_landmark_files/16069/100015657.parquet,16069,100015657,cloud
3,train_landmark_files/25571/1000210073.parquet,25571,1000210073,bird
4,train_landmark_files/62590/1000240708.parquet,62590,1000240708,owie


In [40]:
with open("Kaggle_Data/sign_to_prediction_index_map.json", 'r') as f:
    sign_to_prediction_index_map = json.load(f)

In [41]:
landmark_data = pd.read_parquet("Kaggle_Data/train_landmark_files/16069/100015657.parquet")
landmark_data.head()

,frame,row_id,type,landmark_index,x,y,z
0,103,103-face-0,face,0,0.437886,0.437599,-0.051134
1,103,103-face-1,face,1,0.443258,0.392901,-0.067054
2,103,103-face-2,face,2,0.443997,0.409998,-0.042990
3,103,103-face-3,face,3,0.435256,0.362771,-0.039492
4,103,103-face-4,face,4,0.443780,0.381762,-0.068013


In [ ]:
one_frame = landmark_data[landmark_data["frame"] == 103]
for landmark_type, type_data in one_frame.groupby("type"):
    print(landmark_type,  type_data.shape)

print(f"Total Frames - {len(landmark_data['frame'].unique())}")

face (468, 7)
left_hand (21, 7)
pose (33, 7)
right_hand (21, 7)
Total Frame - 105


In [13]:
def motion_features(positions, fps):
    """
    positions: numpy array of shape (T, 3) -> [ [x0,y0,z0], [x1,y1,z1], ... ]
    fps: frames per second (float)
    """
    positions = np.asarray(positions, dtype=float)
    T = positions.shape[0]
    dt = 1.0 / fps

    # 1. Displacement between consecutive frames
    disp = positions[1:] - positions[:-1]        # shape: (T-1, 3)
    dx, dy, dz = disp[:, 0], disp[:, 1], disp[:, 2]

    # 2. Distance & speed
    dist = np.linalg.norm(disp, axis=1)         # (T-1,)
    speed = dist / dt                           # (T-1,)

    # Avoid division by zero
    eps = 1e-8
    safe_dist = np.maximum(dist, eps)

    # 3. Unit direction
    direction = disp / safe_dist[:, None]       # (T-1, 3)

    # 4. Yaw & Pitch
    yaw = np.arctan2(dy, dx)                    # (T-1,)
    pitch = np.arctan2(dz, np.sqrt(dx**2 + dy**2))

    # 5. Axis angles (optional)
    alpha = np.arccos(dx / safe_dist)           # angle w.r.t X
    beta  = np.arccos(dy / safe_dist)           # angle w.r.t Y
    gamma = np.arccos(dz / safe_dist)           # angle w.r.t Z

    # 6. Turning angle between consecutive displacements
    v1 = disp[:-1]
    v2 = disp[1:]
    dot = np.sum(v1 * v2, axis=1)
    denom = (np.linalg.norm(v1, axis=1) * np.linalg.norm(v2, axis=1)) + eps
    cos_theta = np.clip(dot / denom, -1.0, 1.0)
    turning_angle = np.arccos(cos_theta)        # (T-2,)

    # 7. Yaw / Pitch change
    def wrap_angle(a):
        # wrap to (-pi, pi]
        return (a + np.pi) % (2 * np.pi) - np.pi

    delta_yaw = wrap_angle(yaw[1:] - yaw[:-1])      # (T-2,)
    delta_pitch = pitch[1:] - pitch[:-1]            # (T-2,)

    # 8. Velocity & acceleration
    velocity = disp / dt                         # (T-1, 3)
    vel_mag = np.linalg.norm(velocity, axis=1)   # (T-1,)

    acc = (velocity[1:] - velocity[:-1]) / dt    # (T-2, 3)
    acc_mag = np.linalg.norm(acc, axis=1)        # (T-2,)

    # 9. Jerk (optional)
    jerk = (acc[1:] - acc[:-1]) / dt             # (T-3, 3)
    jerk_mag = np.linalg.norm(jerk, axis=1)      # (T-3,)

    return {
        # "displacement": disp,        # (T-1, 3)
        "distance": dist,            # (T-1,)
        "speed": speed,              # (T-1,)
        # "direction": direction,      # (T-1, 3)
        "yaw": yaw,                  # (T-1,)
        "pitch": pitch,              # (T-1,)
        "axis_angle_x": alpha,       # (T-1,)
        "axis_angle_y": beta,        # (T-1,)
        "axis_angle_z": gamma,       # (T-1,)
        # "turning_angle": turning_angle,   # (T-2,)
        # "delta_yaw": delta_yaw,           # (T-2,)
        # "delta_pitch": delta_pitch,       # (T-2,)
        # "velocity": velocity,             # (T-1, 3)
        "velocity_mag": vel_mag,          # (T-1,)
        # "acceleration": acc,              # (T-2, 3)
        # "acceleration_mag": acc_mag,      # (T-2,)
        # "jerk": jerk,                     # (T-3, 3)
        # "jerk_mag": jerk_mag,             # (T-3,)
    }


In [34]:
landmark_consolidated = dict()

for landmark_type, landmark_type_data in landmark_data.groupby("type"):
    
    if landmark_type == "face":
        continue

    data_consolidation = []

    for landmark_id, landmark_id_data in landmark_type_data.groupby("landmark_index"):

        coordinates = landmark_id_data[['x', 'y', 'z']].values
        features_dict = motion_features(coordinates, 30)

        features_array = []
        for _, array in features_dict.items():
            features_array.append(array)

        features_array = np.array(features_array)
        data_consolidation.append(features_array)

    landmark_consolidated[landmark_type] = np.array(data_consolidation)

# landmark_consolidated = np.array(landmark_consolidated)
# landmark_consolidated.shape


In [36]:
for k,v in landmark_consolidated.items():
    x,y,z = v.shape
    w = v.transpose(2,0,1).reshape(z, x*y)
    landmark_consolidated[k] = w
    

for k,v in landmark_consolidated.items():
    print(k)
    print(v.shape)

left_hand
(104, 168)
pose
(104, 264)
right_hand
(104, 168)


In [37]:
final_data = np.hstack(list(landmark_consolidated.values()))
final_data.shape

(104, 600)

In [18]:
landmark_consolidated

array([dict_values([array([0.10908342, 0.09286445, 0.07847106, ..., 0.06209686, 0.0514153 ,
              0.02941432], shape=(2204,)), array([3.27250259, 2.78593352, 2.35413185, ..., 1.86290574, 1.54245889,
              0.88242947], shape=(2204,)), array([-2.56141223, -2.7190512 , -3.10885682, ...,  2.94106419,
               2.33408318,  2.04109991], shape=(2204,)), array([ 0.03670169, -0.11216785, -0.21333962, ..., -0.64065319,
               0.17060608,  0.61710399], shape=(2204,)), array([2.56038557, 2.7052849 , 2.92579386, ..., 2.47452702, 2.32028411,
              1.94934861], shape=(2204,)), array([2.15053543, 1.99051403, 1.60278976, ..., 1.41042022, 0.77834355,
              0.75683438], shape=(2204,)), array([1.53409464, 1.68296418, 1.78413595, ..., 2.21144952, 1.40019025,
              0.95369233], shape=(2204,)), array([3.27250259, 2.78593352, 2.35413185, ..., 1.86290574, 1.54245889,
              0.88242947], shape=(2204,))])                                                

In [48]:
landmark_consolidated = np.asarray(landmark_consolidated).reshape(1, 105, 150)
landmark_consolidated.shape

(1, 105, 150)

In [41]:
parquet_files_path = Path(
    r"Kaggle_Data/train_landmark_files"
)

for signer_folder in parquet_files_path.iterdir():
    if signer_folder.name == '.DS_Store':
        continue

    for parquet_file in signer_folder.iterdir():

        signing_word = train_data[
            (train_data['participant_id'] == int(parquet_file.parent.name)) & 
            (train_data['sequence_id'] == int(parquet_file.stem))
        ]['sign'].iloc[0]

        signing_word_id = sign_to_prediction_index_map[signing_word]

In [64]:
prediction = np.zeros(len(sign_to_prediction_index_map))
prediction[signing_word_id] = 1
prediction = prediction.reshape(1, -1)

In [50]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [59]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMClassifier, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return self.softmax(out)

In [55]:
input_size = landmark_consolidated.shape[2]
hidden_size = 64
num_layers = 2
num_classes = len(sign_to_prediction_index_map)

model = LSTMClassifier(input_size, hidden_size, num_layers, num_classes)
model.to(device)

model.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [68]:
X_tensor = torch.tensor(landmark_consolidated, dtype=torch.float32).to(device)
Y_tensor = torch.tensor(prediction, dtype=torch.float32).to(device)

In [70]:
num_epochs = 10

for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = criterion(outputs, Y_tensor)
    loss.backward()
    optimizer.step()
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}')



Epoch [1/10], Loss: 5.723202228546143
Epoch [2/10], Loss: 5.6187520027160645
Epoch [3/10], Loss: 5.518866539001465
Epoch [4/10], Loss: 5.417169570922852
Epoch [5/10], Loss: 5.308719158172607
Epoch [6/10], Loss: 5.189612865447998
Epoch [7/10], Loss: 5.056641101837158
Epoch [8/10], Loss: 4.907191276550293
Epoch [9/10], Loss: 4.739643573760986
Epoch [10/10], Loss: 4.553947448730469
